# Hakawi voice demo — Kaggle GPU

This notebook embeds the current `deployment/lightning-tts` service, model revisions,
voices and smoke tests. No repository clone, payment method, or tunnel account is needed.
The first setup downloads several GB. No model weights are embedded in this notebook.

**Before running:** Create/import a private Kaggle notebook. In Session options enable
**Internet** and select **GPU T4 ×2** (or a newer NVIDIA GPU). Only GPU 0 is used;
two T4s do not combine their memory. Complete Kaggle phone verification if GPU/Internet
controls require it. Check your available quota in the UI.

Run cells 1–7 in order. Cell 8 tests custom voice cloning. Cells 9–10 are manual
diagnostics/stop controls: **do not Run All**, because the final cell stops the demo.
Keep the interactive session running while recording/submitting your live demo.
Kaggle sessions and the temporary URL expire; this is not permanent hosting.
Outbound tunnel access and GPU availability must be confirmed in your actual session.
Do not bypass Kaggle session limits or use browser keep-alive scripts.


## 1. Unpack the embedded service
Run again safely in the same session. The original server is copied without edits.


In [ ]:
import hashlib
import json
import pathlib
import subprocess
import sys
ROOT = pathlib.Path('/kaggle/working/hakawi-tts')
ROOT.mkdir(parents=True, exist_ok=True)
previous_demo = globals().get('demo')
if previous_demo and any(p.poll() is None for p in previous_demo.processes.values()):
    raise RuntimeError('A demo is running. Stop it before unpacking/setup again.')

FILES = {'server.py': '"""Hakawi\'s backend-compatible Lightning TTS endpoint: POST /."""\nimport base64\nimport hashlib\nimport io\nimport json\nimport logging\nimport os\nimport time\nfrom collections import OrderedDict\nfrom pathlib import Path\n\nimport litserve as ls\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom omnivoice import OmniVoice\n\nfrom audio_utils import decode_base64_audio, normalize_audio, sanitize_name\n\nBASE_DIR = Path(__file__).resolve().parent\nMODEL_DIR = BASE_DIR / "models" / "lahgtna-omnivoice-v2"\nBUILTIN_DIR = BASE_DIR / "pre_existing_voices"\nMAX_TEXT_CHARS = 2000\nMAX_REF_TEXT_CHARS = 2000\nMAX_CACHED_VOICES = 128\nlogger = logging.getLogger("hakawi.tts")\n\n\ndef atomic_write(path: Path, data: bytes) -> None:\n    """Publish one complete file; never expose a partially written manifest."""\n    import tempfile\n    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as stream:\n        temporary = Path(stream.name)\n        stream.write(data)\n    try:\n        os.replace(temporary, path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\nclass HakawiVoiceAPI(ls.LitAPI):\n    def __init__(self):\n        super().__init__(api_path="/", max_batch_size=1)\n\n    def setup(self, device):\n        started = time.perf_counter()\n        if not torch.cuda.is_available():\n            raise RuntimeError("This deployment requires an NVIDIA GPU")\n        for path in [MODEL_DIR / "config.json", MODEL_DIR / "model.safetensors",\n                     MODEL_DIR / "audio_tokenizer" / "model.safetensors"]:\n            if not path.is_file():\n                raise RuntimeError(f"Missing model asset: {path}; run prepare_assets.py")\n        logger.info("Loading OmniVoice on %s", device)\n        self.model = OmniVoice.from_pretrained(\n            str(MODEL_DIR), device_map=device, dtype=torch.float16,\n            load_asr=False, local_files_only=True,\n        )\n        self.model.eval()\n        self.sample_rate = int(self.model.sampling_rate)\n        self.voice_cache = OrderedDict()\n        self.builtin_refs = {}\n        self.workspace_dir = Path(os.getenv("VOICE_CACHE_DIR", str(BASE_DIR / "runtime_voices"))).resolve()\n        self.workspace_dir.mkdir(parents=True, exist_ok=True)\n        for transcript in sorted(BUILTIN_DIR.glob("*.txt")):\n            key = sanitize_name(transcript.stem)\n            audio_path = transcript.with_suffix(".wav")\n            if not audio_path.is_file():\n                raise RuntimeError(f"Missing built-in voice audio: {audio_path}")\n            ref_text = self._text(transcript.read_text(encoding="utf-8"), "ref_text", MAX_REF_TEXT_CHARS)\n            self.builtin_refs[key] = (audio_path, ref_text)\n            self._cache_prompt(key, self._create_prompt(audio_path, ref_text))\n        if not self.builtin_refs:\n            raise RuntimeError("No built-in voices; run prepare_assets.py before deployment")\n        # Warm inference kernels before LitServe declares the worker ready.\n        if os.getenv("WARMUP", "1") == "1":\n            key = "am-othman" if "am-othman" in self.builtin_refs else next(iter(self.builtin_refs))\n            with torch.inference_mode():\n                self.model.generate(text="أهلاً بيك في حكاوي.", voice_clone_prompt=self._get_prompt(key))\n        logger.info("Ready: %d built-in voices, sample_rate=%d, startup=%.2fs",\n                    len(self.builtin_refs), self.sample_rate, time.perf_counter() - started)\n\n    @staticmethod\n    def _text(value, field, limit):\n        if not isinstance(value, str) or not value.strip():\n            raise ValueError(f"{field} must be a non-empty string")\n        value = value.strip()\n        if len(value) > limit:\n            raise ValueError(f"{field} exceeds the {limit}-character limit")\n        return value\n\n    def _create_prompt(self, audio_path, ref_text):\n        with torch.inference_mode():\n            prompt = self.model.create_voice_clone_prompt(ref_audio=str(audio_path), ref_text=ref_text)\n        # Cache reference tokens on CPU so many custom voices do not consume GPU VRAM.\n        prompt.ref_audio_tokens = prompt.ref_audio_tokens.detach().cpu()\n        return prompt\n\n    def _cache_prompt(self, key, prompt):\n        self.voice_cache[key] = prompt\n        self.voice_cache.move_to_end(key)\n        while len(self.voice_cache) > MAX_CACHED_VOICES:\n            self.voice_cache.popitem(last=False)\n\n    def _get_prompt(self, key):\n        if key in self.voice_cache:\n            self.voice_cache.move_to_end(key)\n            return self.voice_cache[key]\n        # A custom manifest, if present, is a complete locally saved reference.\n        manifest = self.workspace_dir / f"{key}.json"\n        if manifest.is_file():\n            metadata = json.loads(manifest.read_text(encoding="utf-8"))\n            audio_path = self.workspace_dir / metadata["audio_file"]\n            if audio_path.parent.resolve() != self.workspace_dir:\n                raise ValueError("Invalid cached audio path")\n            ref_text = self._text(metadata["ref_text"], "ref_text", MAX_REF_TEXT_CHARS)\n        elif key in self.builtin_refs:\n            audio_path, ref_text = self.builtin_refs[key]\n        else:\n            return None\n        prompt = self._create_prompt(audio_path, ref_text)\n        self._cache_prompt(key, prompt)\n        return prompt\n\n    def _save_and_cache(self, key, audio_b64, ref_text):\n        ref_text = self._text(ref_text, "ref_text", MAX_REF_TEXT_CHARS)\n        wav = normalize_audio(decode_base64_audio(audio_b64), self.sample_rate)\n        # Content-addressed audio plus an atomic manifest keeps voice updates consistent.\n        digest = hashlib.sha256(wav + ref_text.encode("utf-8")).hexdigest()[:20]\n        audio_path = self.workspace_dir / f"{key}-{digest}.wav"\n        atomic_write(audio_path, wav)\n        prompt = self._create_prompt(audio_path, ref_text)\n        metadata = json.dumps({"audio_file": audio_path.name, "ref_text": ref_text}, ensure_ascii=False)\n        atomic_write(self.workspace_dir / f"{key}.json", metadata.encode("utf-8"))\n        self._cache_prompt(key, prompt)\n        return prompt\n\n    def decode_request(self, request):\n        return request\n\n    def predict(self, data):\n        started = time.perf_counter()\n        key = "unknown"\n        try:\n            if not isinstance(data, dict):\n                raise ValueError("Request body must be a JSON object")\n            action = data.get("action")\n            if action not in {"save_character", "generate"}:\n                raise ValueError("action must be \'generate\' or \'save_character\'")\n            key = sanitize_name(data.get("char_name"))\n            if action == "save_character":\n                self._save_and_cache(key, data.get("audio_prompt"), data.get("ref_text"))\n                return {"status": "success", "message": f"Character \'{key}\' saved!"}\n            text = self._text(data.get("text"), "text", MAX_TEXT_CHARS)\n            prompt = self._get_prompt(key)\n            if prompt is None:\n                if data.get("audio_prompt") and data.get("ref_text"):\n                    prompt = self._save_and_cache(key, data["audio_prompt"], data["ref_text"])\n                else:\n                    # The Hakawi backend detects \'not found\' and resends the reference.\n                    return {"status": "error", "message": f"Character \'{key}\' not found and no reference audio provided."}\n            with torch.inference_mode():\n                generated = self.model.generate(text=text, voice_clone_prompt=prompt)\n            waveform = generated[0] if isinstance(generated, (tuple, list)) else generated\n            if hasattr(waveform, "cpu"):\n                waveform = waveform.detach().cpu().numpy()\n            waveform = np.asarray(waveform, dtype=np.float32)\n            if waveform.ndim != 1 or not waveform.size or not np.isfinite(waveform).all():\n                raise RuntimeError("Model returned an invalid waveform")\n            output = io.BytesIO()\n            sf.write(output, waveform, self.sample_rate, format="WAV", subtype="PCM_16")\n            logger.info("Generated voice=%s characters=%d elapsed=%.2fs", key, len(text), time.perf_counter() - started)\n            return {"status": "success", "audio_base64": base64.b64encode(output.getvalue()).decode("ascii")}\n        except ValueError as exc:\n            return {"status": "error", "message": str(exc)}\n        except Exception:\n            logger.exception("TTS failed for voice=%s", key)\n            return {"status": "error", "message": "Speech generation failed; check the deployment logs."}\n\n    def encode_response(self, output):\n        return output\n\n\nif __name__ == "__main__":\n    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(levelname)s %(message)s")\n    server = ls.LitServer(\n        HakawiVoiceAPI(), accelerator="cuda", devices=1, workers_per_device=1,\n        max_payload_size=16 * 1024 * 1024,\n    )\n    # Request timeouts retain LitServe defaults. Hakawi backend is unchanged.\n    server.run(host="0.0.0.0", port=int(os.getenv("PORT", "8000")), generate_client_file=False)\n', 'audio_utils.py': '"""Bounded decoding of browser recordings and reference audio into real WAV."""\nimport base64\nimport binascii\nimport io\nimport os\nimport re\nimport subprocess\n\nimport numpy as np\nimport soundfile as sf\n\nMAX_AUDIO_BYTES = 10 * 1024 * 1024\nMAX_REFERENCE_SECONDS = 30\n\n\ndef sanitize_name(value: str) -> str:\n    if not isinstance(value, str) or len(value) > 100:\n        raise ValueError("char_name must be a string of at most 100 characters")\n    value = value.replace("/", "").replace("\\\\", "").replace("..", "")\n    value = re.sub(r"\\s+", "_", value.strip())\n    value = re.sub(r"[^\\w\\-]", "", value)\n    if not value or value.strip("_-") == "":\n        raise ValueError("char_name contains no usable characters")\n    return value\n\n\ndef decode_base64_audio(value: str) -> bytes:\n    if not isinstance(value, str) or not value:\n        raise ValueError("audio_prompt must contain base64 audio")\n    if len(value) > 4 * ((MAX_AUDIO_BYTES + 2) // 3):\n        raise ValueError("Reference audio exceeds the 10 MiB limit")\n    try:\n        audio = base64.b64decode(value, validate=True)\n    except (binascii.Error, ValueError) as exc:\n        raise ValueError("audio_prompt is not valid base64") from exc\n    if not audio or len(audio) > MAX_AUDIO_BYTES:\n        raise ValueError("Reference audio is empty or exceeds the 10 MiB limit")\n    return audio\n\n\ndef normalize_audio(raw: bytes, sample_rate: int = 24000) -> bytes:\n    """Decode by content, including WebM; filenames never determine the codec."""\n    if not raw or len(raw) > MAX_AUDIO_BYTES:\n        raise ValueError("Reference audio is empty or exceeds the 10 MiB limit")\n    # Bound decoded output to 31 seconds, then reject clips over 30 seconds.\n    command = [\n        os.getenv("FFMPEG_BINARY", "ffmpeg"), "-hide_banner", "-loglevel", "error",\n        "-protocol_whitelist", "pipe", "-i", "pipe:0", "-map", "0:a:0", "-vn",\n        "-ac", "1", "-ar", str(sample_rate),\n        "-t", str(MAX_REFERENCE_SECONDS + 1), "-f", "wav", "pipe:1",\n    ]\n    try:\n        result = subprocess.run(command, input=raw, capture_output=True, check=False)\n    except FileNotFoundError as exc:\n        raise RuntimeError("FFmpeg is missing; install it before starting the server") from exc\n    if result.returncode != 0:\n        raise ValueError("Reference audio cannot be decoded; use WAV, MP3, or WebM")\n    try:\n        waveform, rate = sf.read(io.BytesIO(result.stdout), dtype="float32")\n    except (sf.LibsndfileError, ValueError) as exc:\n        raise ValueError("Reference audio could not be converted to WAV") from exc\n    if waveform.ndim != 1 or waveform.size == 0 or not np.isfinite(waveform).all():\n        raise ValueError("Reference audio is empty or invalid")\n    duration = waveform.size / rate\n    if duration < 0.5 or duration > MAX_REFERENCE_SECONDS:\n        raise ValueError("Reference audio must be between 0.5 and 30 seconds; prefer 3–10 seconds")\n    if float(np.max(np.abs(waveform))) < 1e-5:\n        raise ValueError("Reference audio contains only silence")\n    output = io.BytesIO()\n    sf.write(output, waveform, rate, format="WAV", subtype="PCM_16")\n    return output.getvalue()\n', 'prepare_assets.py': '"""Run online once before deploying; runtime itself uses packaged assets."""\nimport argparse\nimport hashlib\nimport json\nimport urllib.request\nfrom pathlib import Path\n\nfrom huggingface_hub import snapshot_download\nfrom audio_utils import MAX_AUDIO_BYTES, normalize_audio, sanitize_name\n\nBASE_DIR = Path(__file__).resolve().parent\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--voices-only", action="store_true")\n    args = parser.parse_args()\n    versions = json.loads((BASE_DIR / "asset_versions.json").read_text(encoding="utf-8"))\n    model_dir = BASE_DIR / "models" / "lahgtna-omnivoice-v2"\n    if not args.voices_only:\n        for component, destination in [("model", model_dir), ("audio_tokenizer", model_dir / "audio_tokenizer")]:\n            source = versions[component]\n            print(f"Downloading {component} at revision {source[\'revision\']}")\n            snapshot_download(repo_id=source["repo_id"], revision=source["revision"],\n                              local_dir=str(destination), allow_patterns=["*.json", "*.jinja", "*.safetensors"])\n        (model_dir / "asset_versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")\n    voices = json.loads((BASE_DIR / "voices.json").read_text(encoding="utf-8"))\n    voice_dir = BASE_DIR / "pre_existing_voices"\n    voice_dir.mkdir(parents=True, exist_ok=True)\n    downloads = {}\n    checksums = {}\n    for original_key, entry in voices.items():\n        key = sanitize_name(original_key)\n        if entry["url"] not in downloads:\n            with urllib.request.urlopen(entry["url"]) as response:\n                raw = response.read(MAX_AUDIO_BYTES + 1)\n            downloads[entry["url"]] = normalize_audio(raw)\n        wav = downloads[entry["url"]]\n        (voice_dir / f"{key}.wav").write_bytes(wav)\n        (voice_dir / f"{key}.txt").write_text(entry["ref_text"].strip(), encoding="utf-8")\n        checksums[key] = hashlib.sha256(wav).hexdigest()\n        print(f"Prepared {key}: mono PCM WAV")\n    (voice_dir / "checksums.json").write_text(json.dumps(checksums, indent=2), encoding="utf-8")\n    print("Assets ready. Run python preflight.py on a GPU next.")\n\n\nif __name__ == "__main__":\n    main()\n', 'preflight.py': '"""Check the deployment environment before loading multi-gigabyte weights."""\nimport shutil\nimport subprocess\nimport sysconfig\nfrom pathlib import Path\n\nimport soundfile as sf\nimport torch\nfrom omnivoice import OmniVoice\n\n\ndef main():\n    base = Path(__file__).resolve().parent\n    if not shutil.which("ffmpeg") or not shutil.which("gcc"):\n        raise SystemExit("Install ffmpeg and gcc first")\n    if not (Path(sysconfig.get_path("include")) / "Python.h").is_file():\n        raise SystemExit("Python.h is missing for this Python interpreter; install matching development headers")\n    if not torch.cuda.is_available():\n        raise SystemExit("CUDA unavailable: choose a GPU machine and check the NVIDIA driver")\n    # Exercise CUDA, rather than only checking the presence of a driver.\n    probe = torch.ones(16, device="cuda")\n    assert float((probe * 2).sum().item()) == 32\n    print("GPU:", torch.cuda.get_device_name(0))\n    print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)\n    if not hasattr(OmniVoice, "create_voice_clone_prompt"):\n        raise SystemExit("Installed OmniVoice does not support prepared voice prompts")\n    model = base / "models" / "lahgtna-omnivoice-v2"\n    for relative in ["config.json", "model.safetensors", "tokenizer.json",\n                     "audio_tokenizer/config.json", "audio_tokenizer/model.safetensors",\n                     "audio_tokenizer/preprocessor_config.json"]:\n        if not (model / relative).is_file():\n            raise SystemExit(f"Missing {relative}; run prepare_assets.py")\n    transcripts = list((base / "pre_existing_voices").glob("*.txt"))\n    if not transcripts:\n        raise SystemExit("No built-in voices; run prepare_assets.py")\n    for transcript in transcripts:\n        info = sf.info(transcript.with_suffix(".wav"))\n        if info.format != "WAV" or info.samplerate != 24000 or info.channels != 1:\n            raise SystemExit(f"Invalid prepared voice: {transcript.stem}")\n    subprocess.run(["gcc", "--version"], stdout=subprocess.DEVNULL, check=True)\n    print(f"Preflight passed: {len(transcripts)} voices, model + audio tokenizer available")\n\n\nif __name__ == "__main__":\n    main()\n', 'smoke_test.py': '"""Test the exact JSON contract used by Hakawi, without printing credentials."""\nimport argparse\nimport base64\nimport io\nimport json\nimport os\nimport time\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\n\nimport soundfile as sf\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--url", default=os.getenv("VOICE_API_URL", "http://127.0.0.1:8000"))\n    parser.add_argument("--character", default="am-othman")\n    parser.add_argument("--text", default="أهلاً بيك في حكاوي، صوت الماضي حي في الحاضر.")\n    parser.add_argument("--inline-audio", type=Path)\n    parser.add_argument("--ref-text")\n    parser.add_argument("--save-only", action="store_true")\n    parser.add_argument("--output", type=Path, default=Path("test-output/speech.wav"))\n    args = parser.parse_args()\n    if bool(args.inline_audio) != bool(args.ref_text):\n        parser.error("--inline-audio and --ref-text must be supplied together")\n    if args.save_only and not args.inline_audio:\n        parser.error("--save-only requires --inline-audio and --ref-text")\n    headers = {"Content-Type": "application/json"}\n    token = os.getenv("VOICE_API_KEY")\n    if token:\n        headers["Authorization"] = f"Bearer {token}"\n    health_request = urllib.request.Request(args.url.rstrip("/") + "/health", headers=headers)\n    with urllib.request.urlopen(health_request) as response:\n        print("Health:", response.status)\n    payload = {"action": "save_character" if args.save_only else "generate",\n               "char_name": args.character, "text": args.text}\n    if args.inline_audio:\n        payload.update(audio_prompt=base64.b64encode(args.inline_audio.read_bytes()).decode("ascii"),\n                       ref_text=args.ref_text)\n    request = urllib.request.Request(args.url.rstrip("/") + "/", headers=headers,\n                                    data=json.dumps(payload).encode("utf-8"), method="POST")\n    started = time.perf_counter()\n    # No explicit HTTP timeout: keep the user\'s current timeout policy.\n    with urllib.request.urlopen(request) as response:\n        result = json.load(response)\n    if result.get("status") != "success":\n        raise SystemExit(f"TTS failed: {result.get(\'message\', result)}")\n    if args.save_only:\n        print(result["message"])\n        return\n    audio = base64.b64decode(result["audio_base64"], validate=True)\n    info = sf.info(io.BytesIO(audio))\n    if info.format != "WAV" or info.frames == 0:\n        raise SystemExit("Server returned invalid or empty WAV audio")\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_bytes(audio)\n    print(f"Success: {args.output}, {info.samplerate} Hz, {info.duration:.2f}s audio, "\n          f"{time.perf_counter() - started:.2f}s request")\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except urllib.error.HTTPError as exc:\n        raise SystemExit(f"HTTP {exc.code}: check endpoint authentication and deployment logs") from exc\n', 'requirements.txt': '# Install the CUDA build explicitly; keep torch/torchaudio matched.\n--extra-index-url https://download.pytorch.org/whl/cu128\ntorch==2.8.0+cu128\ntorchaudio==2.8.0+cu128\nomnivoice==0.2.1\nlitserve==0.2.19\ntransformers==5.18.0\naccelerate==1.15.0\nhuggingface-hub==2.1.1\nnumpy==2.3.5\nsoundfile==0.14.0\n', 'asset_versions.json': '{\n  "model": {\n    "repo_id": "oddadmix/lahgtna-omnivoice-v2",\n    "revision": "55c38a613e316499829eef60b006edc6da9499e2"\n  },\n  "audio_tokenizer": {\n    "repo_id": "eustlb/higgs-audio-v2-tokenizer",\n    "revision": "528e871c2a26c4f0f7773b9754e2e1acae20899d"\n  }\n}\n', 'voices.json': '{\n  "am-othman": {\n    "url": "https://hueymfgudrgdlmyaxeoi.supabase.co/storage/v1/object/public/characters/audios/am-othman.wav.mp3",\n    "ref_text": "لهجة الصعيد لهجة واعرة جوي مش أي حد يتكلمها"\n  },\n  "am-mohamed": {\n    "url": "https://hueymfgudrgdlmyaxeoi.supabase.co/storage/v1/object/public/characters/audios/aswan.wav.mp3",\n    "ref_text": "ولا في حد نتوَنَّس معاه الناس زمان البتحَكَّى الحكاوي الحلوة دي احسن من الكلام بتاع هنا الشباب اديلو يومين ولا فاهمينه"\n  },\n  "amr-abdeen-modern": {\n    "url": "https://hueymfgudrgdlmyaxeoi.supabase.co/storage/v1/object/public/characters/audios/amr-abdeen-modern.mp3",\n    "ref_text": "كرروا موضوع الهجاء الحرفي ده مئات المرات على أسماء الملوك  و المدن اللي زي رمسيس و أحمس و طيبة و كيمت"\n  },\n  "amr-abdeen": {\n    "url": "https://hueymfgudrgdlmyaxeoi.supabase.co/storage/v1/object/public/characters/audios/Ancient.wav.mp3",\n    "ref_text": "واع ثِن خِيِمِت فِي دُو دِي يو سِيسُو سِيفِخو خيمينو بِسيج"\n  },\n  "ramsis": {\n    "url": "https://hueymfgudrgdlmyaxeoi.supabase.co/storage/v1/object/public/characters/audios/Ancient.wav.mp3",\n    "ref_text": "واع ثِن خِيِمِت فِي دُو دِي يو سِيسُو سِيفِخو خيمينو بِسيج"\n  },\n  "khufu": {\n    "url": "https://hueymfgudrgdlmyaxeoi.supabase.co/storage/v1/object/public/characters/audios/khufu.mp3",\n    "ref_text": "متنساش بردو يا فندم ان اي توتر في باب المندب أو مضيق هرمز بيعلي تكلفة الشحن و بيخوف المستثمر الاجنبي فا بيسحب فلوسه فورا"\n  },\n  "الهرم الأكبر — خوفو": {\n    "url": "https://hueymfgudrgdlmyaxeoi.supabase.co/storage/v1/object/public/characters/audios/khufu.mp3",\n    "ref_text": "متنساش بردو يا فندم ان اي توتر في باب المندب أو مضيق هرمز بيعلي تكلفة الشحن و بيخوف المستثمر الاجنبي فا بيسحب فلوسه فورا"\n  }\n}\n', 'tests/test_server.py': '"""CPU tests use real audio decoding and a fake GPU model; no model download."""\nimport base64\nimport contextlib\nimport importlib.util\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport tempfile\nimport types\nimport unittest\nfrom unittest.mock import patch\n\nimport numpy as np\nimport soundfile as sf\n\nBASE = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(BASE))\nfrom audio_utils import decode_base64_audio, normalize_audio, sanitize_name\n\n\ndef wav_bytes(seconds=1, rate=24000, channels=1):\n    wave = (np.sin(np.arange(int(rate * seconds)) * 2 * np.pi * 220 / rate) * 0.15).astype(np.float32)\n    if channels == 2:\n        wave = np.column_stack([wave, wave])\n    out = io.BytesIO()\n    sf.write(out, wave, rate, format="WAV", subtype="PCM_16")\n    return out.getvalue()\n\n\nclass FakeTokens:\n    def detach(self):\n        return self\n\n    def cpu(self):\n        return self\n\n\nclass FakeModel:\n    sampling_rate = 22050  # Different rate proves the output is not hardcoded.\n\n    def __init__(self):\n        self.prompt_calls = 0\n\n    def create_voice_clone_prompt(self, **kwargs):\n        self.prompt_calls += 1\n        return types.SimpleNamespace(ref_audio_tokens=FakeTokens(), ref_text=kwargs["ref_text"])\n\n    def generate(self, *, text, voice_clone_prompt):\n        return [np.ones(22050, dtype=np.float32) * 0.1]\n\n    def eval(self):\n        return self\n\n\nclass AudioTests(unittest.TestCase):\n    def test_safe_arabic_name_and_empty_names(self):\n        self.assertEqual(sanitize_name("عم عثمان"), "عم_عثمان")\n        for name in ["../", "???", None, "a" * 101]:\n            with self.assertRaises(ValueError):\n                sanitize_name(name)\n\n    def test_base64_validation(self):\n        for audio in ["%%%%", "", "data:audio/wav;base64,abcd"]:\n            with self.assertRaises(ValueError):\n                decode_base64_audio(audio)\n\n    def test_real_wav_normalization(self):\n        wav = normalize_audio(wav_bytes(rate=44100, channels=2))\n        info = sf.info(io.BytesIO(wav))\n        self.assertEqual((info.format, info.channels, info.samplerate), ("WAV", 1, 24000))\n\n    def test_mp3_and_webm_by_content(self):\n        for codec, container in [("libmp3lame", "mp3"), ("libopus", "webm")]:\n            encoded = subprocess.run(\n                [os.getenv("FFMPEG_BINARY", "ffmpeg"), "-hide_banner", "-loglevel", "error",\n                 "-i", "pipe:0", "-c:a", codec, "-f", container, "pipe:1"],\n                input=wav_bytes(), capture_output=True, check=True,\n            ).stdout\n            normalized = normalize_audio(encoded)\n            self.assertEqual(sf.info(io.BytesIO(normalized)).format, "WAV")\n\n    def test_invalid_long_silent_audio(self):\n        out = io.BytesIO()\n        sf.write(out, np.zeros(24000), 24000, format="WAV")\n        for raw in [b"invalid audio", wav_bytes(seconds=31), out.getvalue()]:\n            with self.assertRaises(ValueError):\n                normalize_audio(raw)\n\n\nclass ServerTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        torch = types.ModuleType("torch")\n        torch.inference_mode = contextlib.nullcontext\n        torch.float16 = "float16"\n        torch.cuda = types.SimpleNamespace(is_available=lambda: True)\n        litserve = types.ModuleType("litserve")\n        class LitAPI:\n            def __init__(self, **kwargs):\n                self.config = kwargs\n        litserve.LitAPI = LitAPI\n        omnivoice = types.ModuleType("omnivoice")\n        omnivoice.OmniVoice = types.SimpleNamespace(from_pretrained=lambda *a, **k: FakeModel())\n        spec = importlib.util.spec_from_file_location("tts_server_under_test", BASE / "server.py")\n        cls.server = importlib.util.module_from_spec(spec)\n        with patch.dict(sys.modules, {"torch": torch, "litserve": litserve, "omnivoice": omnivoice}):\n            spec.loader.exec_module(cls.server)\n\n    def setUp(self):\n        self.temp = tempfile.TemporaryDirectory()\n        self.addCleanup(self.temp.cleanup)\n        root = Path(self.temp.name)\n        model = root / "models"\n        (model / "audio_tokenizer").mkdir(parents=True)\n        for name in ["config.json", "model.safetensors", "audio_tokenizer/model.safetensors"]:\n            (model / name).write_text("test fixture")\n        voices = root / "voices"\n        voices.mkdir()\n        (voices / "am-othman.wav").write_bytes(wav_bytes())\n        (voices / "am-othman.txt").write_text("أهلاً بيك", encoding="utf-8")\n        self.addCleanup(patch.stopall)\n        patch.object(self.server, "MODEL_DIR", model).start()\n        patch.object(self.server, "BUILTIN_DIR", voices).start()\n        patch.dict(os.environ, {"VOICE_CACHE_DIR": str(root / "runtime"), "WARMUP": "0"}).start()\n        self.api = self.server.HakawiVoiceAPI()\n        self.api.setup("cuda:0")\n\n    def generate(self, key="am-othman", **extra):\n        return self.api.predict({"action": "generate", "char_name": key, "text": "أهلاً", **extra})\n\n    def test_builtin_prompt_reused_and_model_sample_rate(self):\n        for _ in range(2):\n            result = self.generate()\n            self.assertEqual(result["status"], "success")\n            info = sf.info(io.BytesIO(base64.b64decode(result["audio_base64"])))\n            self.assertEqual(info.samplerate, 22050)\n        self.assertEqual(self.api.model.prompt_calls, 1)\n\n    def test_missing_voice_preserves_backend_retry_trigger(self):\n        result = self.generate("new_voice")\n        self.assertEqual(result["status"], "error")\n        self.assertIn("not found", result["message"])\n\n    def test_inline_clone_cached_and_restored_from_local_reference(self):\n        result = self.generate("عم عثمان جديد", audio_prompt=base64.b64encode(wav_bytes()).decode(), ref_text="أهلاً")\n        self.assertEqual(result["status"], "success")\n        calls = self.api.model.prompt_calls\n        self.assertEqual(self.generate("عم عثمان جديد")["status"], "success")\n        self.assertEqual(self.api.model.prompt_calls, calls)\n        self.api.voice_cache.clear()\n        self.assertEqual(self.generate("عم عثمان جديد")["status"], "success")\n        self.assertEqual(self.api.model.prompt_calls, calls + 1)\n\n    def test_save_character_action(self):\n        result = self.api.predict({"action": "save_character", "char_name": "test_voice",\n                                   "audio_prompt": base64.b64encode(wav_bytes()).decode(), "ref_text": "أهلاً"})\n        self.assertEqual(result["status"], "success")\n        self.assertEqual(self.generate("test_voice")["status"], "success")\n\n    def test_bad_inputs_and_failed_update_leave_existing_voice_usable(self):\n        for payload in [None, {"action": "generate", "char_name": "../", "text": "hello"},\n                        {"action": "generate", "char_name": "am-othman", "text": "a" * 2001}]:\n            self.assertEqual(self.api.predict(payload)["status"], "error")\n        result = self.api.predict({"action": "save_character", "char_name": "am-othman",\n                                   "audio_prompt": "%%%", "ref_text": "أهلاً"})\n        self.assertEqual(result["status"], "error")\n        self.assertEqual(self.generate()["status"], "success")\n\n    def test_cache_is_bounded_and_evicted_builtin_reloads(self):\n        patch.object(self.server, "MAX_CACHED_VOICES", 2).start()\n        for key in ["one", "two"]:\n            self.assertEqual(self.generate(key, audio_prompt=base64.b64encode(wav_bytes()).decode(), ref_text="أهلاً")["status"], "success")\n        self.assertEqual(len(self.api.voice_cache), 2)\n        self.assertNotIn("am-othman", self.api.voice_cache)\n        self.assertEqual(self.generate()["status"], "success")\n        self.assertEqual(len(self.api.voice_cache), 2)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'gateway.py': '"""Loopback-only Bearer gateway for the temporary Kaggle demo tunnel."""\nfrom __future__ import annotations\n\nimport hmac\nimport http.client\nimport json\nimport os\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nMAX_PAYLOAD = 16 * 1024 * 1024\n\n\ndef make_handler(token: str, upstream_port: int = 8000) -> type[BaseHTTPRequestHandler]:\n    if len(token) < 32 or not token.isascii() or any(char.isspace() for char in token):\n        raise ValueError("VOICE_API_KEY must be at least 32 ASCII characters without whitespace")\n\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self, *_args: object) -> None:\n            pass  # Never log request headers, credentials, text, or reference audio.\n\n        def reply(self, status: int, body: bytes, content_type: str = "application/json") -> None:\n            self.send_response(status)\n            self.send_header("Content-Type", content_type)\n            self.send_header("Content-Length", str(len(body)))\n            self.send_header("Cache-Control", "no-store")\n            self.end_headers()\n            self.wfile.write(body)\n\n        def forward(self) -> None:\n            supplied = self.headers.get("Authorization", "")\n            if not hmac.compare_digest(supplied.encode("utf-8"), f"Bearer {token}".encode("ascii")):\n                self.reply(401, b\'{"detail":"Unauthorized"}\')\n                return\n            if (self.command, self.path) not in {("GET", "/health"), ("POST", "/")}:\n                self.reply(404, b\'{"detail":"Not found"}\')\n                return\n            if self.headers.get("Transfer-Encoding"):\n                self.reply(400, b\'{"detail":"Use Content-Length"}\')\n                return\n            try:\n                length = int(self.headers.get("Content-Length", "0"))\n            except ValueError:\n                self.reply(400, b\'{"detail":"Invalid Content-Length"}\')\n                return\n            if length < 0 or length > MAX_PAYLOAD:\n                self.reply(413, b\'{"detail":"Payload too large"}\')\n                return\n            if self.command == "POST" and not length:\n                self.reply(400, b\'{"detail":"JSON body required"}\')\n                return\n            # Bound stalled uploads without changing inference request timeouts.\n            self.connection.settimeout(30)\n            try:\n                body = self.rfile.read(length)\n            except TimeoutError:\n                self.reply(408, b\'{"detail":"Upload timed out"}\')\n                return\n            if len(body) != length:\n                self.reply(400, b\'{"detail":"Incomplete body"}\')\n                return\n            connection = http.client.HTTPConnection("127.0.0.1", upstream_port, timeout=None)\n            try:\n                connection.request(self.command, self.path, body=body,\n                                   headers={"Content-Type": "application/json"})\n                response = connection.getresponse()\n                self.reply(response.status, response.read(), response.getheader("Content-Type", "application/json"))\n            except (OSError, http.client.HTTPException):\n                self.reply(502, json.dumps({"detail": "Model server unavailable; check notebook logs"}).encode())\n            finally:\n                connection.close()\n\n        do_GET = forward\n        do_POST = forward\n\n    return Handler\n\n\nif __name__ == "__main__":\n    handler = make_handler(os.environ["VOICE_API_KEY"])\n    ThreadingHTTPServer(("127.0.0.1", 8001), handler).serve_forever()\n', 'demo_runtime.py': '"""Notebook process management; no notebook-kernel model imports required."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport signal\nimport subprocess\nimport time\nimport urllib.error\nimport urllib.request\n\nfrom runtime_env import clean_env\n\n\nclass Demo:\n    def __init__(self, root: Path, python: Path, token: str):\n        self.root, self.python, self.token = root, python, token\n        self.processes: dict[str, subprocess.Popen] = {}\n        self.url: str | None = None\n\n    def launch(self, name: str, args: list[str], env: dict[str, str] | None = None) -> None:\n        existing = self.processes.get(name)\n        if existing is not None and existing.poll() is None:\n            raise RuntimeError(f"{name} is already running; stop it first")\n        with (self.root / f"{name}.log").open("w", encoding="utf-8") as log:\n            self.processes[name] = subprocess.Popen(\n                args, cwd=self.root, env=clean_env(env), stdout=log, stderr=subprocess.STDOUT,\n                start_new_session=True,\n            )\n\n    def check(self, name: str) -> None:\n        process = self.processes[name]\n        if process.poll() is not None:\n            raise RuntimeError(f"{name} exited with {process.returncode}. Run the diagnostics cell.")\n\n    def wait_health(self, url: str, name: str, token: str | None = None, seconds: int = 1200) -> None:\n        deadline = time.monotonic() + seconds\n        headers = {"Authorization": f"Bearer {token}"} if token else {}\n        while time.monotonic() < deadline:\n            self.check(name)\n            try:\n                with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=5) as response:\n                    if response.status == 200:\n                        return\n            except (OSError, urllib.error.URLError):\n                pass\n            time.sleep(2)\n        raise RuntimeError(f"{name} did not become healthy within {seconds}s. Run diagnostics.")\n\n    def start_model(self) -> None:\n        env = clean_env()\n        env.update(CUDA_VISIBLE_DEVICES="0", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n                   TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1", PORT="8000",\n                   VOICE_CACHE_DIR=str(self.root / "runtime_voices"), WARMUP="1")\n        env.pop("VOICE_API_KEY", None)\n        self.launch("model", [str(self.python), "server.py"], env)\n        print("Loading model, preparing voices and warming up; allow several minutes...")\n        self.wait_health("http://127.0.0.1:8000/health", "model")\n        print("Model ready.")\n\n    def start_gateway(self) -> None:\n        env = clean_env()\n        env["VOICE_API_KEY"] = self.token\n        self.launch("gateway", [str(self.python), "gateway.py"], env)\n        self.wait_health("http://127.0.0.1:8001/health", "gateway", self.token, 30)\n        self.assert_protected("http://127.0.0.1:8001")\n        print("Bearer gateway ready; unauthenticated requests rejected.")\n\n    @staticmethod\n    def assert_protected(url: str) -> None:\n        for suffix in ["/health", "/"]:\n            request = urllib.request.Request(url + suffix, data=b"{}" if suffix == "/" else None)\n            try:\n                with urllib.request.urlopen(request, timeout=30):\n                    pass\n            except urllib.error.HTTPError as exc:\n                if exc.code == 401:\n                    continue\n                raise RuntimeError(f"Expected 401 from {suffix}, received {exc.code}") from exc\n            raise RuntimeError("Endpoint accepted an unauthenticated request; do not connect backend")\n\n    def start_tunnel(self) -> str:\n        self.check("gateway")\n        binary = self.root / "cloudflared"\n        # Download from the vendor\'s official release and verify GitHub\'s asset digest.\n        request = urllib.request.Request("https://api.github.com/repos/cloudflare/cloudflared/releases/latest",\n                                         headers={"User-Agent": "hakawi-kaggle-demo"})\n        with urllib.request.urlopen(request, timeout=60) as response:\n            release = json.load(response)\n        asset = next(item for item in release["assets"] if item["name"] == "cloudflared-linux-amd64")\n        digest = asset.get("digest", "")\n        if not digest.startswith("sha256:"):\n            raise RuntimeError("Official release has no SHA256 digest; download not executed")\n        expected = digest.split(":", 1)[1]\n        if not binary.is_file() or hashlib.sha256(binary.read_bytes()).hexdigest() != expected:\n            with urllib.request.urlopen(asset["browser_download_url"], timeout=120) as response:\n                content = response.read()\n            if hashlib.sha256(content).hexdigest() != expected:\n                raise RuntimeError("cloudflared checksum mismatch")\n            binary.write_bytes(content)\n            binary.chmod(0o700)\n        print("Tunnel client:", release["tag_name"])\n        self.launch("tunnel", [str(binary), "tunnel", "--url", "http://127.0.0.1:8001",\n                               "--protocol", "http2", "--no-autoupdate"])\n        deadline = time.monotonic() + 90\n        while time.monotonic() < deadline:\n            self.check("tunnel")\n            log = (self.root / "tunnel.log").read_text(encoding="utf-8", errors="replace")\n            match = re.search(r"https://[a-z0-9-]+\\.trycloudflare\\.com", log)\n            if match:\n                self.url = match.group()\n                self.wait_health(self.url + "/health", "tunnel", self.token, 120)\n                self.assert_protected(self.url)\n                print("VOICE_API_URL=" + self.url)\n                print("Public health and authentication checks passed.")\n                return self.url\n            time.sleep(2)\n        raise RuntimeError("Tunnel URL not available; inspect tunnel.log. Kaggle may restrict outbound tunnels.")\n\n    def smoke(self, character: str = "am-othman", public: bool = False, extra: list[str] | None = None) -> Path:\n        if public and not self.url:\n            raise RuntimeError("Start the tunnel first")\n        env = clean_env()\n        env["VOICE_API_KEY"] = self.token\n        output = self.root / "test-output" / f"{\'public\' if public else \'local\'}-{character}.wav"\n        args = [str(self.python), "smoke_test.py", "--url", self.url if public else "http://127.0.0.1:8001",\n                "--character", character, "--output", str(output)]\n        subprocess.run(args + (extra or []), cwd=self.root, env=env, check=True)\n        return output\n\n    def diagnostics(self) -> None:\n        for name, process in self.processes.items():\n            print(f"\\n{name}: {\'running\' if process.poll() is None else \'stopped\'}")\n            log = self.root / f"{name}.log"\n            if log.exists():\n                print("\\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-35:]))\n\n    def stop(self, name: str | None = None) -> None:\n        names = [name] if name else ["tunnel", "gateway", "model"]\n        for item in names:\n            process = self.processes.get(item)\n            if process is not None and process.poll() is None:\n                os.killpg(process.pid, signal.SIGTERM)\n                try:\n                    process.wait(timeout=15)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid, signal.SIGKILL)\n                    process.wait(timeout=15)\n        if name is None or name == "tunnel":\n            self.url = None\n        print("Stopped:", ", ".join(names))\n', 'runtime_env.py': '"""Keep Kaggle\'s injected Python startup paths out of the service environment."""\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\n\n\ndef clean_env(source: dict[str, str] | None = None) -> dict[str, str]:\n    env = dict(os.environ if source is None else source)\n    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "PYTHONUSERBASE"):\n        env.pop(name, None)\n    env["PYTHONNOUSERSITE"] = "1"\n    return env\n\n\ndef prepare_runtime(environment: Path) -> Path:\n    """Create or repair a venv, including one left behind by an interrupted setup."""\n    env = clean_env()\n    executable = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")\n    if not executable.exists():\n        subprocess.run([sys.executable, "-I", "-m", "venv", "--without-pip", str(environment)],\n                       env=env, check=True)\n    probe = subprocess.run([str(executable), "-I", "-m", "pip", "--version"],\n                           env=env, capture_output=True, text=True)\n    if probe.returncode:\n        bootstrap = subprocess.run([str(executable), "-I", "-m", "ensurepip", "--upgrade"],\n                                   env=env, capture_output=True, text=True)\n        if bootstrap.returncode:\n            # The notebook interpreter\'s pip can manage a target interpreter without pip.\n            subprocess.run([sys.executable, "-I", "-m", "pip", "--python", str(executable),\n                            "install", "--upgrade", "pip"], env=env, check=True)\n    subprocess.run([str(executable), "-I", "-m", "pip", "install", "--upgrade", "pip", "wrapt"],\n                   env=env, check=True)\n    return executable\n'}
MANIFEST = {'server.py': 'e617a2af6a5b406ebd468bc94ad6795e89a392c5751f7af7946578e02731c114', 'audio_utils.py': '5ba7401b3a34fa5342e7350d744be58d6d3014cf8b2fca1ee3556f3bb04f52f1', 'prepare_assets.py': 'e8049c9b9db23d3ada355b06a1d22dea4d2ff53dbc71f2642caea04b6037cd9d', 'preflight.py': '1490024b9af7885fb6c412402997462dc541f29659154dee916a2818a3c71344', 'smoke_test.py': 'fca07c4b75c5fed2cd9732cd598c69837f04ac1320c1f6968dfbfcc797290844', 'requirements.txt': '9073c56dd98d22f891265796083b79d5fc26ea4370f60309102c4326f94b4871', 'asset_versions.json': 'd278ec09ed278e4910b834377c03cac73b74ead49df1638b4ec18e9e01f9c02d', 'voices.json': '0484eee250e2927bd75d041c8678718d81ba4cd5f77d6eb838fb88e5daf396d3', 'tests/test_server.py': '8db89e2f9b0812207b057535012848c84651d0d5229b9510366d799a45e84ce1', 'gateway.py': 'f6e1acaff33f09afaf9d38a66731bc9a486a66b65fcef5f1c5de0e0beb4cc6ed', 'demo_runtime.py': '3dff1bc108ee21a06cdcef14b3db66ad10e6f1e6e4d9071e95c028fe1e558b78', 'runtime_env.py': '85c180c8efaad90013156a4cd62c67623dd8fbc197e2c2b12689f62021a9ac28'}

for name, content in FILES.items():
    destination = ROOT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    assert hashlib.sha256(content.encode()).hexdigest() == MANIFEST[name]
    destination.write_text(content, encoding='utf-8')
(ROOT / 'source-manifest.json').write_text(json.dumps(MANIFEST, indent=2))
print('Service unpacked:', ROOT)
print('Model revisions:', json.loads(FILES['asset_versions.json']))


## 2. Install an isolated runtime
The notebook kernel's packages stay separate. Setup uses its own Python environment.
Keep `CUDA_BUILD='cu126'` for Kaggle T4/older drivers. To reproduce Lightning's CUDA
build on a compatible driver, choose `cu128` **before** running this cell.
Both use PyTorch/torchaudio 2.8.0; all other core versions come from Lightning's file.
Stop if this cell fails. Do not continue to later cells with a broken install.


In [ ]:
import shutil
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('This package requires Python 3.11–3.13; inspect Kaggle image settings.')
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Enable an NVIDIA GPU in Kaggle Session options first.')
subprocess.run(['nvidia-smi'], check=True)
if any(not shutil.which(tool) for tool in ['ffmpeg', 'gcc']):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1', 'gcc'], check=True)
CUDA_BUILD = 'cu126'  # cu128 matches Lightning if the driver supports it.
if CUDA_BUILD not in {'cu126', 'cu128'}:
    raise ValueError('Choose cu126 or cu128')
ENV = ROOT / '.venv'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import clean_env, prepare_runtime  # noqa: E402
PYTHON = prepare_runtime(ENV)
def run(*args, env=None):
    subprocess.run([str(PYTHON), *map(str, args)], cwd=ROOT, env=clean_env(env), check=True)
# One resolver transaction pins the matching CUDA pair and current application versions.
requirements = '\n'.join(line for line in FILES['requirements.txt'].splitlines()
                         if line.strip() and not line.startswith(('#', '--', 'torch==', 'torchaudio==')))
requirements += f'\ntorch==2.8.0+{CUDA_BUILD}\ntorchaudio==2.8.0+{CUDA_BUILD}\n'
(ROOT / 'requirements-kaggle.txt').write_text(requirements)
run('-m', 'pip', 'install', '--extra-index-url', 'https://download.pytorch.org/whl/' + CUDA_BUILD,
    '-r', 'requirements-kaggle.txt')
run('-m', 'pip', 'check')
run('-c', "import torch; assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "x=torch.ones(16, device='cuda:0'); assert (x*2).sum().item()==32; "
    "print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))")
with (ROOT / 'installed-versions.txt').open('w') as inventory:
    subprocess.run([str(PYTHON), '-m', 'pip', 'freeze'], stdout=inventory, env=clean_env(), check=True)
print('Isolated runtime ready.')


## 3. Download the model, tokenizer and voice references
Uses the same pinned model revisions and voice URLs as Lightning. Public assets normally
need no Hugging Face token. If access is gated for your account, add an optional Kaggle
Secret called `HF_TOKEN` and enable notebook access to it. Never paste tokens into code.


In [ ]:
asset_env = clean_env()
asset_env.pop('HF_HUB_OFFLINE', None)
asset_env.pop('TRANSFORMERS_OFFLINE', None)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = None  # Optional; public model downloads do not require this secret.
if hf_token:
    asset_env['HF_TOKEN'] = hf_token
run('prepare_assets.py', env=asset_env)
hf_token = None
asset_env.pop('HF_TOKEN', None)
run('preflight.py')
run('-m', 'unittest', 'discover', '-s', 'tests', '-v')
print('Assets, CUDA operation and CPU service tests passed.')


## 4. Set the demo key and start the service
Create a Kaggle Secret named **VOICE_API_KEY** containing at least 32 ASCII characters
without spaces, and enable access for this notebook. You can instead enter that key
in the hidden prompt. Save the same value privately for your backend configuration.
The notebook never prints it. Startup loads voices and performs a warmup generation.


In [ ]:
import getpass
try:
    from kaggle_secrets import UserSecretsClient
    voice_key = UserSecretsClient().get_secret('VOICE_API_KEY')
except Exception:
    voice_key = getpass.getpass('Demo key (32+ ASCII characters, no spaces): ')
if len(voice_key) < 32 or not voice_key.isascii() or any(c.isspace() for c in voice_key):
    raise ValueError('Use at least 32 ASCII characters without whitespace.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from demo_runtime import Demo
previous_demo = globals().get('demo')
if previous_demo:
    previous_demo.stop()
demo = Demo(ROOT, PYTHON, voice_key)
voice_key = None
demo.start_model()
demo.start_gateway()


## 5. Generate and listen locally
Both character names below are configured in the current deployment. Check pronunciation,
identity and clarity yourself. A valid WAV alone does not prove voice quality.


In [ ]:
from IPython.display import Audio, display
for character in ['am-othman', 'khufu']:
    audio_path = demo.smoke(character)
    display(Audio(filename=str(audio_path)))


## 6. Create a temporary HTTPS URL and test it
Uses Cloudflare Quick Tunnel, without an account or payment method. Only the protected
gateway is exposed. The client binary is verified against the official release digest.
HTTP/2 is used because notebook networks may restrict QUIC/UDP.
The URL changes when you restart the tunnel. Outbound TCP port 7844 must be available.
If tunneling fails, inspect diagnostics; no permanent URL or uptime is promised.


In [ ]:
VOICE_API_URL = demo.start_tunnel()
public_audio = demo.smoke('am-othman', public=True)
display(Audio(filename=str(public_audio)))
print('Copy this URL into your backend:', VOICE_API_URL)
print('Use your private VOICE_API_KEY value from cell 4. It is intentionally not printed.')


## 7. Connect Hakawi and record the submission demo
In your **backend hosting environment**, set these values (do not put them in the frontend):

```dotenv
VOICE_API_URL="COPY_THE_HTTPS_URL_FROM_CELL_6"
VOICE_API_KEY="YOUR_SAME_PRIVATE_KEY_FROM_CELL_4"
```

For a local backend, edit `backend/.env` and restart the backend. For a hosted backend,
update its environment variables and restart/redeploy it. Existing `generate` and
`save_character` request/response formats are preserved. No frontend code change is needed.
Test a built-in character in Hakawi, then test one custom recording. The backend's existing
save-character request has a 10-second timeout: prepare custom demo voices with cell 8 if
first-time cloning exceeds that window. Generation retains its current timeout behavior.

Record and download your submission video while the session is live. Keep a local video
copy because a notebook session is not a persistent deployed service. Stop the session
when finished. A new Kaggle session needs setup again and a new tunnel URL; it may also
need custom voice references resent. Do not publish saved notebook outputs containing
personal recordings. Quotas/session availability are controlled by Kaggle.


## 8. Optional — verify save-character, reuse and inline cloning
Run once before the demo to verify the same three operations your backend uses.
This uses the included am-othman sample to create two disposable demo voice names.


In [ ]:
reference = ROOT / 'pre_existing_voices/am-othman.wav'
transcript = (ROOT / 'pre_existing_voices/am-othman.txt').read_text(encoding='utf-8')
inline_args = ['--inline-audio', str(reference), '--ref-text', transcript]
demo.smoke('kaggle-saved-demo', public=True, extra=inline_args + ['--save-only'])
display(Audio(filename=str(demo.smoke('kaggle-saved-demo', public=True))))
display(Audio(filename=str(demo.smoke('kaggle-inline-demo', public=True, extra=inline_args))))
print('Save, saved-voice reuse and inline cloning passed.')


## 9. Diagnostics — run manually when needed
Logs report model failures and tunnel connection problems. The current server can log
character identifiers. Review logs before sharing them. Never print `demo.token` or env vars.
If only the tunnel stopped, run `demo.stop('tunnel')`, then rerun cell 6.
For a failed model, stop the demo with cell 10, correct the reported problem and rerun cell 4.


In [ ]:
demo.diagnostics()
subprocess.run(['nvidia-smi'], check=False)


## 10. Stop — run manually after recording
Stops only the processes started by this notebook. Also stop the Kaggle session in its UI
to release your GPU quota. Files under /kaggle/working are session storage; download any
needed WAVs/video separately. Do not save a model/voice output bundle publicly.


In [ ]:
demo.stop()


## References
- [Kaggle Notebook settings](https://www.kaggle.com/docs/notebooks)
- [Kaggle GPU usage](https://www.kaggle.com/docs/efficient-gpu-usage)
- [Cloudflare Quick Tunnels and limitations](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)
- [PyTorch 2.8 CUDA builds](https://pytorch.org/get-started/previous-versions/)

Cloudflare Quick Tunnels have no uptime SLA, a 200 concurrent-request limit and no SSE.
This API uses ordinary JSON responses. Long generations may still encounter proxy limits;
use short demo responses and measure the public request time in cell 6.
